# RF Board Loopback Demo (Remote Version)

This notebook is a remote-friendly rewrite of the on-board QICK RF board demo
([demo_03_tprocv2-full.ipynb](https://github.com/sarafs1926/tprocv2_demos/blob/main/rfboard_docs/qick_rf216/demo_03_tprocv2-full.ipynb)).
The RF board has two programmable 30 dB attenuators for each DAC channel and one for each ADC
channel, plus a programmable ADMV8818 bandpass filter on every DAC and ADC channel. The loopback
programs below check that the attenuators and filters operate properly.

**Note: the original demo assumed 60 dB of in-line attenuation between the DAC and ADC channels in loopback.**

The original notebook runs *on* the ZCU216 itself. Here everything runs from a remote PC through
the Pyro4 nameserver, using the wrappers in `helpers/rfboard.py` (see
[README_rfboard.md](../README_rfboard.md)).

## Differences from the on-board demo

1. **Connection**: the demo loads a bitstream with `RFQickSoc216V1(...)` on the board. Remotely,
   the firmware is loaded by the Pyro server on the board; we just connect a proxy through the
   nameserver. Only *methods* are callable over the proxy (`soc.rfb_set_gen_filter(...)`,
   `soc.rfb_set_bias(...)`, `soc.clear_interrupts()`, program execution). Attribute access
   (`soc.gens`, `soc.dac_cards`, `soc.rf`, `soc.filter_spi`) is **not** available.
2. **Introspection**: the demo enumerates DAC/ADC cards and chain pinouts (`soc.dac_cards`,
   `gen.rfb_ch.global_ch`, `GpioMCP23S08` card IDs). None of that works remotely — the
   channel ↔ pinout mapping must come from the
   [RF216 wiki table](https://github.com/openquantumhardware/qick/wiki/RF216-(RF-board-for-ZCU216))
   and is recorded in the config YAML.
3. **Units**: the `helpers/rfboard.py` functions take **MHz** (the config convention); the raw
   `soc.rfb_*` methods take **GHz**.
4. **Filter granularity**: remotely we can select lowpass / highpass / bandpass / bypass by
   frequency (qick picks the ADMV8818 band and capacitor state from `fc`), but we cannot address
   individual band indices or 4-bit capacitor states — those require direct register writes on
   the board.
5. **Saturation handling**: `soc.clear_interrupts(error_on_interrupt=False)` works remotely but
   gives no per-interrupt-mask breakdown, and the ADC DSA attenuator read/set (`adc.DSA`) is
   board-only.
6. **Added by this repo**: config-driven defaults, `rfboard_active` state tracking, and
   `activate_qubit_rf()` for switching shared channels between qubits — none of which exist in
   the demo.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import os
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

from qick import QickConfig
# tProc v2 classes: programs compile on this PC and run on the board over Pyro
from qick.asm_v2 import AveragerProgramV2, QickSweep1D

from slab_qick_calib.exp_handling.instrumentmanager import InstrumentManager
from slab_qick_calib.helpers import config, rfboard

## Connection

The nameserver must already be running on this PC (`start_nameserver.bat`) and the board's Pyro
server must have the RF-board firmware loaded. This replaces the demo's

```python
soc = RFQickSoc216V1('/home/xilinx/jupyter_notebooks/fw/.../qick_216_rfb.bit', clk_output=None)
```

In [ ]:
cfg_file = 'sample_50_rfboard.yml'
cfg_path = os.path.join(os.getcwd(), '..', 'configs', cfg_file)
auto_cfg = config.load(cfg_path)

# Port 8888 is required — the nameserver listens there, not the default 9090
im = InstrumentManager(ns_address=auto_cfg['aliases']['ip'], port=8888)
soc = im[auto_cfg['aliases']['soc']]
soccfg = QickConfig(soc.get_cfg())
print(soccfg)

## Hardware configuration / channel lookup

**Board-only.** The demo builds the firmware-channel ↔ RF-board-pinout lookup table by walking
hardware objects, which requires attribute access on the local SoC object and so cannot run over
the Pyro proxy:

```python
# >>> runs only on the board itself <<<
for iCard, card in enumerate(soc.dac_cards):
    if isinstance(card, DacRfCard216):
        print("DAC card %d: RF" % iCard)
        print("filters:", [chain.read_filter("CHIPTYPE") for chain in card.chains])

for i, gen in enumerate(soc.gens):
    if isinstance(gen.rfb_ch, Chain216):
        print("DAC ch %d in QICK firmware = RF board pinout %d" % (i, gen.rfb_ch.global_ch))

for i, buf in enumerate(soc.avg_bufs):
    if isinstance(buf.rfb_ch, Chain216):
        print("ADC ch %d in QICK firmware = RF board pinout %d" % (i, buf.rfb_ch.global_ch))

for card_num in range(8):
    soc.board_sel.enable(board_id=card_num)
    gpio = GpioMCP23S08(soc.filter_spi, ch_en=4 if card_num < 4 else 2, dev_addr=0, iodir=0xf0)
    print("card %d: ID %d" % (card_num, gpio.read_reg("GPIO_REG") >> 4))
```

Remotely, the mapping comes from the
[RF216 wiki table](https://github.com/openquantumhardware/qick/wiki/RF216-(RF-board-for-ZCU216))
and is recorded once in the config YAML (`hw.soc.adcs.readout.ch`, `hw.soc.dacs.*.ch`). We read
the channels for one qubit's readout chain and use them as the loopback pair:

In [ ]:
hw = auto_cfg.hw.soc
print("ADC readout channels per qubit:", hw.adcs.readout.ch)
print("DAC readout channels per qubit:", hw.dacs.readout.ch)
if hasattr(hw.dacs, 'qubit'):
    print("DAC qubit-drive channels per qubit:", hw.dacs.qubit.ch)

# Loopback test pair (QICK firmware numbering) — taken from the config, override as needed
qi = 0
GEN_CH = hw.dacs.readout.ch[qi]  # DAC
RO_CH = hw.adcs.readout.ch[qi]   # ADC
print(f"loopback pair: GEN_CH={GEN_CH}, RO_CH={RO_CH}")

## Bias DACs

The RF board has 8 bias DACs (-10 to 10 V). `rfb_set_bias()` and `rfb_get_bias()` are methods,
so they work unchanged over the proxy. The helper `rfboard.set_dc_bias()` is the config-driven
version: it reads `dc_ch`/`dc_val` from `hw.soc.dacs.flux` for the given qubit.

In [ ]:
print("read bias:", soc.rfb_get_bias(0))
soc.rfb_set_bias(0, 1.0)
print("read bias:", soc.rfb_get_bias(0))
soc.rfb_set_bias(0, 0.0)

# Config-driven version: sets the flux bias channel for qubit qi to its stored dc_val
rfboard.set_dc_bias(qi, soc, auto_cfg)

## Loopback program

Unchanged from the demo — tProc v2 programs compile on this PC and execute on the board over
Pyro, so no rewrite is needed.

In [ ]:
class LoopbackProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        self.declare_gen(ch=gen_ch, nqz=cfg['nqz'], mixer_freq=cfg['mixer_freq'], ro_ch=ro_ch)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])
        self.add_readoutconfig(ch=ro_ch, name="myro",
                               freq=cfg['freq'],
                               gen_ch=gen_ch,
                               outsel='product')
        self.add_cosine(ch=gen_ch, name="ramp", length=cfg['ramp_len'], even_length=True)
        self.add_pulse(ch=gen_ch, name="mypulse", ro_ch=ro_ch,
                       style="const",
                       freq=cfg['freq'],
                       length=cfg['flat_len'],
                       phase=cfg['phase'],
                       gain=cfg['gain'],
                       )
        self.send_readoutconfig(ch=cfg['ro_ch'], name="myro", t=0)

    def _body(self, cfg):
        self.delay_auto()
        self.pulse(ch=cfg['gen_ch'], name="mypulse", t=0.0)
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'], mr=True)

## Testing the programmable attenuators

The demo programs the filters and attenuators with four raw calls (frequencies in **GHz**):

```python
soc.rfb_set_gen_filter(gen_ch, fc=freq/1000, ftype='bandpass', bw=1.0)
soc.rfb_set_ro_filter(ro_ch, fc=freq/1000, ftype='bandpass', bw=1.0)
soc.rfb_set_gen_rf(gen_ch, 5, 15)   # two 30 dB DAC attenuators
soc.rfb_set_ro_rf(ro_ch, 30)        # one 30 dB ADC attenuator
```

The helper `rfboard.set_bandpass_rf()` does all of this in one call (in **MHz**), validates the
band edges against the ADMV8818's 2–18 GHz span, waits 30 ms for the chain to settle, and clears
sticky interrupt flags. Decreasing attenuation on either side should increase the I/Q amplitudes,
and vice versa.

In [ ]:
prog_cfg = {'gen_ch': GEN_CH,
            'ro_ch': RO_CH,
            'mixer_freq': 3500,
            'freq': 4000,
            'nqz': 2,
            'trig_time': 0.0,
            'ro_len': 3.0,
            'flat_len': 2.0,
            'ramp_len': 1.0,
            'phase': 0,
            'gain': 1.0
            }

# One call: both bandpass filters + all three attenuators + settle + clear_interrupts
rfboard.set_bandpass_rf(soc, GEN_CH, RO_CH, fc_MHz=prog_cfg['freq'], bw_MHz=1000,
                        tx_att1=5, tx_att2=15, ro_att=30)

prog = LoopbackProgram(soccfg, reps=1, final_delay=0.5, cfg=prog_cfg)
iq_list = prog.acquire_decimated(soc, soft_avgs=100)
t = prog.get_time_axis(ro_index=0)
iq = iq_list[0]
plt.plot(t, iq[:, 0], label="I value")
plt.plot(t, iq[:, 1], label="Q value")
plt.legend()
plt.ylabel("amplitude [ADU]")
plt.xlabel("time [us]");

## Frequency sweeping

`FreqSweepProgram` sweeps the pulse frequency over `steps` points using `QickSweep1D`. Unchanged
from the demo apart from the filter/attenuator setup going through the helper.

In [ ]:
class FreqSweepProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']

        self.declare_gen(ch=gen_ch, nqz=cfg['nqz'], mixer_freq=cfg['mixer_freq'], ro_ch=ro_ch)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        self.add_loop("myloop", self.cfg["steps"])
        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['freq'], gen_ch=gen_ch)

        self.add_pulse(ch=gen_ch, name="mypulse", ro_ch=ro_ch,
                       style="const",
                       freq=cfg['freq'],
                       length=cfg['pulse_len'],
                       phase=cfg['phase'],
                       gain=cfg['gain'],
                       )

    def _body(self, cfg):
        # if you delay the config by too long, you can see the readout get reconfigured
        # in the middle of your pulse
        self.send_readoutconfig(ch=cfg['ro_ch'], name="myro", t=0)
        self.pulse(ch=cfg['gen_ch'], name="mypulse", t=0)
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])


# do a sweep with 5 points and plot decimated
steps = 5
start_freq = 3900
end_freq = 4100
prog_cfg = {'steps': steps,
            'gen_ch': GEN_CH,
            'ro_ch': RO_CH,
            'freq': QickSweep1D("myloop", start_freq, end_freq),
            'mixer_freq': 4000,
            'nqz': 2,
            'trig_time': 0.4,
            'ro_len': 0.3,
            'pulse_len': 0.2,
            'phase': 0,
            'gain': 1.0
            }

rfboard.set_bandpass_rf(soc, GEN_CH, RO_CH, fc_MHz=prog_cfg['mixer_freq'], bw_MHz=1000,
                        tx_att1=5, tx_att2=15, ro_att=30)

prog = FreqSweepProgram(soccfg, reps=1, final_delay=0.5, cfg=prog_cfg)
iq_list = prog.acquire_decimated(soc, soft_avgs=100)
t = prog.get_time_axis(ro_index=0)

sweep_freqs = np.linspace(start_freq, end_freq, steps)
for ii, iq in enumerate(iq_list[0]):
    plt.plot(t, np.abs(iq.dot([1, 1j])), label="step = {}, f = {} MHz".format(ii, int(sweep_freqs[ii])))

plt.legend()
plt.ylabel("amplitude [ADU]")
plt.xlabel("time [us]");

## Transmission efficiency

`measure_s21()` measures the forward transmission coefficient ($S_{21}$) between the DAC and ADC
channels by stitching together DDS-bandwidth-sized frequency sweeps at stepped mixer frequencies.
Unchanged from the demo — it only uses the client-side `soccfg` and remote program execution.

In [ ]:
def measure_s21(gen_ch, ro_ch, nqz, gain, steps=101, dds_range=0.45, overlap=0, plot=False, progress=True):
    prog_cfg = {'steps': steps,
                'gen_ch': gen_ch,
                'ro_ch': ro_ch,
                'nqz': nqz,
                'trig_time': 0.4,
                'pulse_len': 10.0,
                'ro_len': 10.1,
                'phase': 0,
                'gain': gain
                }
    allfreqs = []
    allpowers = []
    f_dds = soccfg['gens'][gen_ch]['f_dds']
    mixer_freqs = np.arange(0.5, 10000/f_dds, dds_range*2-overlap)*f_dds
    for i, mixer_freq in enumerate(tqdm(mixer_freqs, disable=not progress)):
        prog_cfg['mixer_freq'] = mixer_freq
        prog_cfg['freq'] = QickSweep1D("myloop", mixer_freq-dds_range*f_dds, mixer_freq+dds_range*f_dds)

        prog = FreqSweepProgram(soccfg, reps=10, final_delay=1.0, cfg=prog_cfg)
        freqs = prog.get_pulse_param('myro', 'freq', as_array=True)
        iq_list = prog.acquire(soc, soft_avgs=1, progress=False)
        iq_complex = iq_list[0][0].dot([1, 1j])

        mags = np.abs(iq_complex)
        powers = 20*np.log10(mags)
        allfreqs.append(freqs)
        allpowers.append(powers)
        if plot:
            plt.plot(freqs, powers, label="mixer_freq=%f" % (mixer_freq))
            plt.ylabel("S21 [arb. dB]")
            plt.xlabel("Frequency [MHz]")
    allfreqs = np.array(allfreqs).flatten()
    allpowers = np.array(allpowers).flatten()
    return allfreqs, allpowers

## $S_{21}$ vs frequency

With the filters bypassed, the broadband response of the RF board is visible. The demo bypassed
each channel with raw `rfb_set_*_filter(..., ftype='bypass')` calls; the helper
`rfboard.bypass_all_filters()` does the same for lists of channels (and can record the state in
the config). There is no helper for setting attenuators alone, so those stay as raw calls.

In [ ]:
rfboard.bypass_all_filters(soc, [GEN_CH], [RO_CH])
soc.rfb_set_gen_rf(GEN_CH, 5, 15)
soc.rfb_set_ro_rf(RO_CH, 30)
measure_s21(GEN_CH, RO_CH, 2, 1.0, steps=501, dds_range=0.45, overlap=0.1, plot=True);

## $S_{21}$ at different Nyquist zones

Transmittance can be optimized at lower frequencies ($\le f_s/2$, first Nyquist zone) with
`nqz=1` or at higher frequencies with `nqz=2`. The DAC sample rate is read from `soccfg` instead
of being hardcoded.

In [ ]:
rfboard.bypass_all_filters(soc, [GEN_CH], [RO_CH])
soc.rfb_set_gen_rf(GEN_CH, 5, 15)
soc.rfb_set_ro_rf(RO_CH, 30)

fig, ax = plt.subplots(figsize=(8, 6))
plt.plot(*measure_s21(GEN_CH, RO_CH, nqz=1, gain=1.0), label="nqz=1")
plt.plot(*measure_s21(GEN_CH, RO_CH, nqz=2, gain=1.0), label="nqz=2")
plt.ylim(bottom=0)
ylims = ax.get_ylim()

# visualizing the Nyquist zones
fs_dac = soccfg['gens'][GEN_CH]['fs']  # DAC sample rate in MHz
nyquist_zones = ["1st Nyquist Zone", "2nd Nyquist Zone", "3rd Nyquist Zone"]
for nindex in range(len(nyquist_zones)):
    zone = nindex*fs_dac/2
    ax.fill_between([zone, zone + fs_dac/2], min(ylims), max(ylims),
                    label=nyquist_zones[nindex], alpha=0.35)

ax.set_ylim(ylims[0], ylims[1])
ax.set_xlim(0, 3/2 * fs_dac)
ax.set_xticks(np.arange(0, 1.51*fs_dac, fs_dac/2), ["0", r"$f_s/2$", r"$f_s$", r"$3f_s/2$"])
plt.ylabel("S21 [arb. dB]")
plt.xlabel("Frequency [MHz]")
plt.legend();

## Filtering with the RF board

Each DAC and ADC channel has one programmable ADMV8818 filter: four high-pass and four low-pass
filter bands, each with a 4-bit tunable capacitor giving 16 states, plus a bypass path on each
side. See the
[ADMV8818 datasheet](https://www.analog.com/media/en/technical-documentation/data-sheets/admv8818-ep.pdf).

**Board-only.** The demo addresses bands and capacitor states directly with register writes,
which needs attribute access to the local chain objects:

```python
# >>> runs only on the board itself <<<
def set_filter(gen_ch, ro_ch, lpf, hpf, state=0):
    sw = 0xc0 + (hpf << 3) + lpf
    filt_bits = (state << 4) + state
    rfb_ch = soc.gens[gen_ch].rfb
    rfb_ch.brd_sel.enable(rfb_ch.rfboard_ch)
    rfb_ch.filter.reg_wr('WR0_SW', sw)
    rfb_ch.filter.reg_wr('WR0_FILTER', filt_bits)
    rfb_ch.brd_sel.disable()
    # ... same for soc.avg_bufs[ro_ch].rfb
```

Remotely we cannot pick a band index or capacitor state, but `rfb_set_gen_filter()` /
`rfb_set_ro_filter()` accept `ftype='lowpass'` and `'highpass'` in addition to `'bandpass'` and
`'bypass'`, and qick chooses the band and state from the requested cutoff `fc` (in GHz). So
instead of sweeping band/state indices, we sweep the cutoff frequency.

### Low-pass filtering

Sweep the low-pass cutoff and watch the upper edge of the $S_{21}$ response move.

In [ ]:
soc.rfb_set_gen_rf(GEN_CH, 5, 15)
soc.rfb_set_ro_rf(RO_CH, 30)

for fc in tqdm([2.0, 4.0, 6.0, 8.0]):  # cutoff in GHz (raw rfb_* calls take GHz)
    soc.rfb_set_gen_filter(GEN_CH, fc=fc, ftype='lowpass')
    soc.rfb_set_ro_filter(RO_CH, fc=fc, ftype='lowpass')
    plt.plot(*measure_s21(GEN_CH, RO_CH, 2, 1.0, progress=False), label="fc=%.1f GHz" % fc)

rfboard.bypass_all_filters(soc, [GEN_CH], [RO_CH])
plt.ylabel("S21 [arb. dB]")
plt.xlabel("Frequency [MHz]")
plt.ylim(bottom=0)
plt.legend();

### High-pass filtering

Same idea with `ftype='highpass'`: the lower edge of the response moves with the cutoff.

In [ ]:
soc.rfb_set_gen_rf(GEN_CH, 5, 15)
soc.rfb_set_ro_rf(RO_CH, 30)

for fc in tqdm([2.0, 4.0, 6.0, 8.0]):  # cutoff in GHz
    soc.rfb_set_gen_filter(GEN_CH, fc=fc, ftype='highpass')
    soc.rfb_set_ro_filter(RO_CH, fc=fc, ftype='highpass')
    plt.plot(*measure_s21(GEN_CH, RO_CH, 2, 1.0, progress=False), label="fc=%.1f GHz" % fc)

rfboard.bypass_all_filters(soc, [GEN_CH], [RO_CH])
plt.ylabel("S21 [arb. dB]")
plt.xlabel("Frequency [MHz]")
plt.ylim(bottom=0)
plt.legend();

## Bandpass filtering

Bandpass at a fixed center with decreasing bandwidth. `set_bandpass_rf()` replaces the raw
filter + attenuator calls and additionally warns if the band edges leave the ADMV8818's usable
2–18 GHz span.

In [ ]:
freq = 6000  # MHz
for bw in tqdm([1500, 1000, 500, 200, 100]):
    rfboard.set_bandpass_rf(soc, GEN_CH, RO_CH, fc_MHz=freq, bw_MHz=bw,
                            tx_att1=5, tx_att2=15, ro_att=30, verbose=False)
    plt.plot(*measure_s21(GEN_CH, RO_CH, 2, 1.0, progress=False), label="bw=%d MHz" % bw)

plt.ylabel("S21 [arb. dB]")
plt.xlabel("Frequency [MHz]")
plt.ylim(bottom=0)
plt.legend();

## Testing saturation in loopback

If the ZCU216 receives too large a pulse it saturates, raises an interrupt, and automatically
adds 15 dB of attenuation via its built-in ADC DSA. Two interrupts matter:

- `XRFDC_ADC_OVR_RANGE_MASK`: the ADC counts hit full scale. Does **not** add attenuation.
- `XRFDC_ADC_OVR_VOLTAGE_MASK`: the input voltage exceeded the ADC safe range. **This adds
  15 dB of attenuation**, visible as an abrupt amplitude drop mid-pulse.

**Board-only.** The demo's diagnostic tooling needs `import xrfdc` and `soc.rf` attribute
access, neither of which exists remotely:

```python
# >>> runs only on the board itself <<<
import xrfdc

def clear_interrupts(soc, verbose=False):
    # walks soc['adcs'], reads XRFdc_GetIntrStatus per tile/block, decodes the
    # interrupt masks by name, and clears them with XRFdc_IntrClr (up to 5 passes)
    ...

def print_attenuator(soc, ro_ch):
    tile, block = [int(x) for x in soc['readouts'][ro_ch]['adc']]
    print(soc.rf.adc_tiles[tile].blocks[block].DSA)

def set_attenuator(soc, ro_ch, val):
    tile, block = [int(x) for x in soc['readouts'][ro_ch]['adc']]
    soc.rf.adc_tiles[tile].blocks[block].DSA['Attenuation'] = val
```

(See the original demo for the full interrupt-mask table.)

Remotely we can still call `soc.clear_interrupts(error_on_interrupt=False)`, which clears the
sticky flags but does not report which masks were set, and we cannot read back or reset the DSA —
if the board has tripped its +15 dB attenuation, reset it on the board (or restart the Pyro
server). Watch the I/Q trace instead: a clean flat-top pulse means no saturation; an abrupt
mid-pulse amplitude drop means the over-voltage interrupt fired.

In [ ]:
class LoopbackProgramFlatTop(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        self.declare_gen(ch=gen_ch, nqz=cfg['nqz'], mixer_freq=cfg['mixer_freq'], ro_ch=ro_ch)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])
        self.add_readoutconfig(ch=ro_ch, name="myro",
                               freq=cfg['freq'],
                               gen_ch=gen_ch,
                               outsel='product')
        self.add_cosine(ch=gen_ch, name="ramp", length=cfg['ramp_len'], even_length=True)
        self.add_pulse(ch=gen_ch, name="mypulse", ro_ch=ro_ch,
                       style="flat_top",
                       envelope="ramp",
                       freq=cfg['freq'],
                       length=cfg['flat_len'],
                       phase=cfg['phase'],
                       gain=cfg['gain'],
                       )
        self.send_readoutconfig(ch=cfg['ro_ch'], name="myro", t=0)

    def _body(self, cfg):
        self.delay_auto()
        self.pulse(ch=cfg['gen_ch'], name="mypulse", t=0.0)
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'], mr=True)


prog_cfg = {'gen_ch': GEN_CH,
            'ro_ch': RO_CH,
            'mixer_freq': 3000,
            'freq': 3000,
            'nqz': 1,
            'trig_time': 0.0,
            'ro_len': 3.0,
            'flat_len': 1.0,
            'ramp_len': 1.0,
            'phase': 0,
            'gain': 1.0
            }

rfboard.set_bandpass_rf(soc, GEN_CH, RO_CH, fc_MHz=prog_cfg['freq'], bw_MHz=1000,
                        tx_att1=0, tx_att2=10, ro_att=25)

prog = LoopbackProgramFlatTop(soccfg, reps=1, final_delay=0.5, cfg=prog_cfg)
iq_list = prog.acquire_decimated(soc, soft_avgs=1)

# clears sticky flags; unlike the on-board version, no per-mask report is available
soc.clear_interrupts(error_on_interrupt=False)

t = prog.get_time_axis(ro_index=0)
iq = iq_list[0]
plt.plot(t, iq[:, 0], label="I value")
plt.plot(t, iq[:, 1], label="Q value")
plt.plot(t, np.abs(iq.dot((1, 1j))), label="magnitude")
plt.legend()
plt.ylabel("amplitude [ADU]")
plt.xlabel("time [us]");

## Beyond the demo: config-driven RF state

Everything above programmed the hardware with explicit values. This repo's normal workflow
stores per-qubit filter/attenuator/bias settings in the config YAML and tracks what was last
programmed on each physical channel in the `rfboard_active` section. The single call most users
need is `activate_qubit_rf()`, which switches all shared channels to one qubit's stored settings
(see [README_rfboard.md](../README_rfboard.md) for details):

In [ ]:
# Switch all shared RF channels (ADC, readout DAC, qubit DAC, DC bias) to qubit qi's
# stored settings; cfg_file= reloads the YAML first and persists the resulting state
auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)

# Query what is currently programmed on each physical channel
rfboard.get_active_rf_state(auto_cfg)